# This is a notebook for *ongoing* MoM data analysis

#### installs

In [ ]:
%pip install \
    requests \
    beautifulsoup4 \
    pandas \
    matplotlib \
    seaborn

#### Config (connect to Caddy server)

In [13]:
import getpass

BASE_URL = input("Enter Base URL: ").strip()
ADMIN = input("Enter Username: ").strip()
PASS = getpass.getpass("Enter Password: ")

AUTH = (ADMIN, PASS)

print("done")

done


## understanding the folder structure

In [15]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

visited_dirs = set()
session = requests.Session()
session.auth = AUTH

def generate_folder_tree(url, indent=""):
    if not url.endswith('/'):
        url += '/'
    
    if url in visited_dirs:
        return
    visited_dirs.add(url)
    
    try:
        response = session.get(url, timeout=5)
        if response.status_code != 200:
            return
            
        soup = BeautifulSoup(response.text, "html.parser")
        sub_dirs = []
        
        for link in soup.find_all("a"):
            href = link.get("href", "")
            text = link.text.strip()
            
            if not href or href.startswith("?") or text in [".", "..", "../", "Parent Directory"]:
                continue
                
            if href.endswith("/") or text.endswith("/"):
                sub_dirs.append((href.rstrip("/"), urljoin(url, href)))
                
        sub_dirs.sort(key=lambda x: x[0])
        
        for i, (name, full_url) in enumerate(sub_dirs):
            is_last = (i == len(sub_dirs) - 1)
            marker = "└── " if is_last else "├── "
            
            print(f"{indent}{marker}{name}/")
            generate_folder_tree(full_url, indent + ("    " if is_last else "│   "))
            
    except Exception:
        pass

print("Generating directory structure tree...\n")
print("root/")
generate_folder_tree(BASE_URL)
print("\nTree generation complete.")

Generating directory structure tree...

root/
├── ./DFO/
│   ├── ../
│   ├── ./DFO_MoM/
│   │   ├── ../
│   │   └── ../../
│   ├── ./DFO_image/
│   │   ├── ../
│   │   └── ../../
│   └── ./DFO_summary/
│       ├── ../
│       └── ../../
├── ./Final_Alert/
│   └── ../
├── ./GFMS/
│   ├── ../
│   ├── ./GFMS_MoM/
│   │   ├── ../
│   │   └── ../../
│   ├── ./GFMS_image/
│   │   ├── ../
│   │   └── ../../
│   └── ./GFMS_summary/
│       ├── ../
│       └── ../../
├── ./GLOFAS/
│   └── ../
├── ./HWRF/
│   ├── ../
│   ├── ./HWRF_MoM/
│   │   ├── ../
│   │   └── ../../
│   ├── ./HWRF_image/
│   │   ├── ../
│   │   └── ../../
│   └── ./HWRF_summary/
│       ├── ../
│       └── ../../
└── ./VIIRS/
    ├── ../
    ├── ./VIIRS_MoM/
    │   ├── ../
    │   └── ../../
    ├── ./VIIRS_image/
    │   ├── ../
    │   └── ../../
    └── ./VIIRS_summary/
        ├── ../
        └── ../../

Tree generation complete.


## File format/name/size/count/coverage

In [20]:
import re
import sys
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

all_target_folders = [
    "/Final_Alert/",
    "/DFO/DFO_MoM/", "/DFO/DFO_image/", "/DFO/DFO_summary/",
    "/GFMS/GFMS_MoM/", "/GFMS/GFMS_image/", "/GFMS/GFMS_summary/",
    "/GLOFAS/",
    "/HWRF/HWRF_MoM/", "/HWRF/HWRF_image/", "/HWRF/HWRF_summary/",
    "/VIIRS/VIIRS_MoM/", "/VIIRS/VIIRS_image/", "/VIIRS/VIIRS_summary/"
]

file_profiles = []
grand_total_files = 0
grand_total_bytes = 0

def parse_size_to_bytes(size_str):
    """Converts structural size values (e.g. '14.2 MB') into integer bytes."""
    if not size_str:
        return 0
    size_str = size_str.strip().upper().replace(",", "")
    match = re.search(r'([\d.]+)\s*([KMGT]?B|[BKMGT])', size_str)
    if not match:
        return 0
    value = float(match.group(1))
    unit = match.group(2)
    multipliers = {
        'B': 1, 'KB': 1024, 'MB': 1024**2, 'GB': 1024**3, 'TB': 1024**4,
        'K': 1024, 'M': 1024**2, 'G': 1024**3
    }
    return int(value * multipliers.get(unit, 1))

def format_bytes_to_human(size_in_bytes):
    """Converts calculated file metrics back to human-readable strings."""
    if pd.isna(size_in_bytes) or size_in_bytes is None or size_in_bytes <= 0:
        return "0.00 B"
    size_in_bytes = float(size_in_bytes)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_in_bytes < 1024.0:
            return f"{size_in_bytes:.2f} {unit}"
        size_in_bytes /= 1024.0
    return f"{size_in_bytes:.2f} PB"

total_folders = len(all_target_folders)
print(f"🤖 Scanning {total_folders} folders to extract granular pattern-level matrices...\n")

for index, folder_path in enumerate(all_target_folders, start=1):
    target_url = f"{BASE_URL.rstrip('/')}{folder_path}"
    print(f"[{index}/{total_folders}] Auditing track: {folder_path} ...", end="\r", flush=True)
    
    try:
        response = requests.get(target_url, auth=AUTH, timeout=15)
        if response.status_code != 200:
            file_profiles.append({
                'folder': folder_path, 'pattern': "Offline/Unreachable", 
                'size': 0, 'date': pd.NaT, 'is_empty': True
            })
            continue
            
        soup = BeautifulSoup(response.text, "html.parser")
        found_any_valid_files = False
        
        for row in soup.find_all(["tr", "li"]):
            link = row.find("a")
            if not link:
                continue
                
            href = link.get("href", "")
            filename = link.text.strip()
            
            # 1. Clean out standard Apache/Nginx web interface navigation anchors
            if (not filename or filename.startswith("?") or 
                filename in [".", "..", "../", "Parent Directory"] or 
                "caddyserver.com" in href or "caddyserver.com" in filename):
                continue
                
            # Ignore nested subdirectory links within this execution scope
            if href.endswith("/") or filename.endswith("/"):
                continue

            # 2. Hard filter out non-data payloads
            if not filename.lower().endswith(('.csv', '.geojson', '.tiff', '.tif', '.nc')):
                continue
            
            found_any_valid_files = True
            
            # Pattern conversion logic from script 1
            pattern_str = re.sub(r'\d{10}', 'YYYYMMDDHH', filename)
            pattern_str = re.sub(r'\d{8}', 'YYYYMMDD', pattern_str)
            pattern_str = re.sub(r'\d{4}[-_]\d{2}[-_]\d{2}', 'YYYY-MM-DD', pattern_str)
            
            # Size extraction logic from script 2
            size_bytes = 0
            all_cols = [col.get_text().strip() for col in row.find_all(["td", "span"]) if col.get_text().strip()]
            for item in all_cols:
                if re.search(r'\b\d+(?:\.\d+)?\s*[KMGT]?B?\b', item, re.IGNORECASE):
                    if not re.search(r'\d{4}-\d{2}-\d{2}', item):
                        parsed_sz = parse_size_to_bytes(item)
                        if parsed_sz > 0:
                            size_bytes = parsed_sz
                            break
            
            # Date checking logic from script 2
            file_date = pd.NaT
            date_match = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', filename)
            if date_match:
                try:
                    file_date = pd.to_datetime(f"{date_match.group(1)}-{date_match.group(2)}-{date_match.group(3)}")
                except ValueError:
                    pass

            file_profiles.append({
                'folder': folder_path,
                'pattern': pattern_str,
                'size': size_bytes,
                'date': file_date,
                'is_empty': False
            })

        if not found_any_valid_files:
            file_profiles.append({
                'folder': folder_path, 'pattern': "N/A (Empty)", 
                'size': 0, 'date': pd.NaT, 'is_empty': True
            })
        
    except Exception as e:
        file_profiles.append({
            'folder': folder_path, 'pattern': "Extraction Failed", 
            'size': 0, 'date': pd.NaT, 'is_empty': True
        })

# Clear status tracking string line
sys.stdout.write('\x1b[2K\r')
sys.stdout.flush()

# Process report data rows
report_rows = []
if file_profiles:
    df_files = pd.DataFrame(file_profiles)
    
    # Core strategy pivot shift: Group by BOTH folder and pattern layout mapping rules
    grouped = df_files.groupby(['folder', 'pattern'])
    
    for (folder_path, pattern_str), group in grouped:
        if group['is_empty'].iloc[0]:
            report_rows.append([folder_path, "0", pattern_str, "N/A", "N/A", "N/A"])
            continue
            
        file_count = len(group)
        grand_total_files += file_count
        
        sizes_series = group['size']
        total_pattern_bytes = sizes_series.sum()
        grand_total_bytes += total_pattern_bytes
        
        tot_sz = format_bytes_to_human(total_pattern_bytes)
        min_sz = format_bytes_to_human(sizes_series.min())
        max_sz = format_bytes_to_human(sizes_series.max())
        avg_sz = format_bytes_to_human(sizes_series.mean())
        size_stats_summary = f"Tot: {tot_sz} | Min: {min_sz} | Max: {max_sz} | Avg: {avg_sz}"
        
        # Timeline logic isolation bound specifically to this file variation subset
        timestamps = group['date'].dropna()
        timeline_range = "N/A"
        health_pct = "N/A"
        
        if len(timestamps) >= 2:
            ts_series = timestamps.drop_duplicates().sort_values()
            start_date = ts_series.min()
            end_date = ts_series.max()
            
            perfect_timeline = pd.date_range(start=start_date, end=end_date, freq='D')
            completeness_pct = (len(ts_series) / len(perfect_timeline)) * 100 if len(perfect_timeline) > 0 else 0
            
            timeline_range = f"{start_date.strftime('%Y-%m-%d')} / {end_date.strftime('%Y-%m-%d')}"
            health_pct = f"{completeness_pct:.1f}%"
            
        report_rows.append([
            folder_path, 
            str(file_count), 
            pattern_str, 
            size_stats_summary, 
            timeline_range, 
            health_pct
        ])

# --- PURE TEXT GRID RENDER ENGINE ---
headers = ["Server Folder Path", "Count", "Discovered Structural File Pattern", "Pattern Size Statistics", "Timeline Range (Start/End)", "Health"]
col_widths = [24, 7, 36, 42, 28, 8]

row_format = "│ " + " │ ".join([f"{{:<{w}}}" for w in col_widths]) + " │"
top_border    = "┌─" + "─┬─".join(["─" * w for w in col_widths]) + "─┐"
header_border = "├─" + "─┼─".join(["─" * w for w in col_widths]) + "─┤"
bottom_border = "└─" + "─┴─".join(["─" * w for w in col_widths]) + "─┘"

print(top_border)
print(row_format.format(*headers))
print(header_border)

# Sort out table visualization layout explicitly by Server Path then Pattern Name
for row in sorted(report_rows, key=lambda x: (x[0], x[2])):
    truncated_row = [str(item)[:w] for item, w in zip(row, col_widths)]
    print(row_format.format(*truncated_row))

print(header_border)
print(row_format.format(
    "GRAND TOTAL DATASETS", 
    str(grand_total_files), 
    "All unique structural layout tracks", 
    f"Total Capacity: {format_bytes_to_human(grand_total_bytes)}", 
    "N/A", 
    "N/A"
))
print(bottom_border + "\n")

🤖 Scanning 14 folders to extract granular pattern-level matrices...

┌──────────────────────────┬─────────┬──────────────────────────────────────┬────────────────────────────────────────────┬──────────────────────────────┬──────────┐
│ Server Folder Path       │ Count   │ Discovered Structural File Pattern   │ Pattern Size Statistics                    │ Timeline Range (Start/End)   │ Health   │
├──────────────────────────┼─────────┼──────────────────────────────────────┼────────────────────────────────────────────┼──────────────────────────────┼──────────┤
│ /DFO/DFO_MoM/            │ 1659    │ Attributes_Clean_YYYYMMDDHHMOM+DFOUp │ Tot: 1.04 GB | Min: 190.00 KB | Max: 952.0 │ 2021-12-15 / 2026-07-10      │ 99.4%    │
│ /DFO/DFO_MoM/            │ 1659    │ Final_Attributes_YYYYMMDDHHMOM+DFOUp │ Tot: 3.53 GB | Min: 712.00 KB | Max: 3.60  │ 2021-12-15 / 2026-07-10      │ 99.4%    │
│ /DFO/DFO_image/          │ 3       │ DFO_YYYYMMDD_Flood_3-Day_250m.tiff   │ Tot: 214.00 MB | Min: 14.00 

In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

# --- Ensure BASE_URL, AUTH, and all_target_folders are defined above this line ---

all_files = []

for folder_path in all_target_folders:
    # Build the full network url target path matching your main framework
    target_url = f"{BASE_URL.rstrip('/')}{folder_path}"
    
    try:
        response = requests.get(target_url, auth=AUTH, timeout=10)
        if response.status_code != 200:
            continue
            
        soup = BeautifulSoup(response.text, "html.parser")
        for link in soup.find_all("a"):
            filename = link.text.strip()
            
            # Find the date string signature
            date_match = re.search(r'_(\d{8})', filename)
            if date_match:
                parsed_date = pd.to_datetime(date_match.group(1), format='%Y%m%d', errors='coerce')
                
                all_files.append({
                    "Folder": folder_path,
                    "File": filename,
                    "ParsedDate": parsed_date
                })
    except Exception:
        pass

# Construct dataframe engine and evaluate absolute earliest dataset bound limits
df_chrono = pd.DataFrame(all_files).dropna(subset=['ParsedDate']).sort_values(by="ParsedDate")

if not df_chrono.empty:
    print(f"📅 ABSOLUTE EARLIEST FILE FOUND:\n"
          f"Folder: {df_chrono.iloc[0]['Folder']}\n"
          f"Filename: {df_chrono.iloc[0]['File']}\n"
          f"True Start Date: {df_chrono.iloc[0]['ParsedDate'].strftime('%Y-%m-%d')}")
else:
    print("No files with valid YYYYMMDD timestamps found.")


## start date

In [ ]:
all_files = [{"Folder": f.replace(BASE_URL, "/"), "File": l.text.strip(), "ParsedDate": pd.to_datetime(re.search(r'_(\d{8})', l.text).group(1), format='%Y%m%d', errors='coerce')} for f in folder_profiles for l in BeautifulSoup(requests.get(f, auth=AUTH).text, "html.parser").find_all("a") if re.search(r'_(\d{8})', l.text)]
df_chrono = pd.DataFrame(all_files).dropna(subset=['ParsedDate']).sort_values(by="ParsedDate")
print(f"📅 ABSOLUTE EARLIEST FILE FOUND:\nFolder: {df_chrono.iloc[0]['Folder']}\nFilename: {df_chrono.iloc[0]['File']}\nTrue Start Date: {df_chrono.iloc[0]['ParsedDate'].strftime('%Y-%m-%d')}") if not df_chrono.empty else print("No files with valid YYYYMMDD timestamps found.")

## visualising data gaps

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("📐 Constructing Inter-Model Conflict & Gap Alignment Matrix...")

# 1. Manually populate the exact timeline profiles extracted from your live server scan
# (Using the exact data from your file output to ensure flawless alignment)
model_gaps = {
    'DFO_MoM':        ['2022-03-17', '2023-11-01', '2023-12-31', '2024-04-14', '2024-12-31'],
    'DFO_summary':    ['2022-03-17', '2023-11-01', '2023-12-31', '2024-04-14', '2024-12-31'],
    'Final_Alert':    ['2022-03-17', '2022-03-18', '2025-10-20', '2025-10-21', '2025-10-22'],
    'GFMS_MoM':       ['2021-12-15', '2025-10-20', '2025-10-21'],
    'GFMS_summary':   ['2025-10-20'],
    'GLOFAS':         ['2025-01-02', '2025-10-17', '2025-10-19'],
    'HWRF_MoM':       ['2022-03-17', '2025-09-09', '2025-10-20', '2025-10-21'],
    'VIIRS_MoM':      ['2021-12-31', '2022-03-17', '2022-11-23', '2023-01-18', '2023-06-02'],
    'VIIRS_summary':  ['2021-12-31', '2022-11-23', '2023-01-18', '2023-06-02', '2023-06-03']
}

# Combine all unique anomaly dates across the entire system
all_anomaly_dates = sorted(list(set([date for dates in model_gaps.values() for date in dates])))

# 2. Build the binary occurrence matrix (1 = Missing/Broken Day, 0 = Healthy Operational Day)
matrix_data = {}
for model, gaps in model_gaps.items():
    matrix_data[model] = [1 if date in gaps else 0 for date in all_anomaly_dates]

df_matrix = pd.DataFrame(matrix_data, index=all_anomaly_dates)

# ==========================================
# VISUALIZE CROSS-MODEL CORRELATION PAIN POINTS
# ==========================================
plt.figure(figsize=(12, 6))
# Create a high-contrast heatmap showing exactly who went offline and when
sns.heatmap(df_matrix.T, cmap="Reds", cbar=False, linewidths=0.5, linecolor="#222")
plt.title("🚨 Critical MoM Failure Matrix: Shared Ingestion Blackout Windows", fontsize=12, fontweight='bold')
plt.xlabel("Specific Calendar Date Points")
plt.ylabel("Subsystem Component Folder")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 3. Log out structural logic flaws directly
print("\n🔍 Automated Consistency & Logic Verification Logic:")
dfo_mom_set = set(model_gaps['DFO_MoM'])
dfo_sum_set = set(model_gaps['DFO_summary'])

if dfo_mom_set == dfo_sum_set:
    print("  • ✅ Consistency: DFO_MoM and DFO_summary feature perfectly mirrored gaps. Pipelines are tied securely.")
else:
    print("  • ❌ Conflict: Inconsistency between DFO MoM output and its Summary metrics timeline.")

# Evaluate the 2025-10-20 system-wide crash programmatically
oct_20_failures = df_matrix.loc['2025-10-20']
critical_count = oct_20_failures.sum()
print(f"  • 🚨 Systemic Failure Identified: On 2025-10-20, {critical_count} critical components crashed simultaneously.")

## File content

In [ ]:
if not df_chrono.empty:
    # 1. Get the earliest file metadata
    earliest_file = df_chrono.iloc[0]
    
    # 2. Reconstruct the full URL (reversing the "/" replacement)
    # Assumes BASE_URL does not end with a trailing slash, or handles it safely
    folder_path = earliest_file['Folder'].lstrip('/')
    file_url = f"{BASE_URL}/{folder_path}/{earliest_file['File']}" if folder_path else f"{BASE_URL}/{earliest_file['File']}"
    
    # 3. Fetch the content
    print(f"\n📥 Fetching contents...")
    response = requests.get(file_url, auth=AUTH)
    
    if response.status_code == 200:
        # 4. Display or process the text content
        print("\n📄 --- FILE CONTENTS ---")
        print(response.text)
        print("------------------------")
    else:
        print(f"❌ Failed to download file. Status code: {response.status_code}")
else:
    print("Cannot read contents because no valid files were found.")


## latest file in each folder

In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Your original list comprehension block
all_files = [
    {
        "Folder": f.replace(BASE_URL, "/"), 
        "File": l.text.strip(), 
        "ParsedDate": pd.to_datetime(re.search(r'_(\d{8})', l.text).group(1), format='%Y%m%d', errors='coerce')
    } 
    for f in folder_profiles 
    for l in BeautifulSoup(requests.get(f, auth=AUTH).text, "html.parser").find_all("a") 
    if re.search(r'_(\d{8})', l.text)
]

df_chrono = pd.DataFrame(all_files).dropna(subset=['ParsedDate']).sort_values(by="ParsedDate")

if not df_chrono.empty:
    # Extract the entry containing the maximum date per unique folder grouping
    df_latest_per_folder = df_chrono.loc[df_chrono.groupby("Folder")["ParsedDate"].idxmax()]
    
    print("\nLATEST FILE FOR EACH FOLDER:")
    print(df_latest_per_folder.to_string(index=False))
else:
    print("No files with valid YYYYMMDD timestamps found.")

## checking for irrelevant folders

In [ ]:
import pandas as pd

# The official documented outage date
OUTAGE_DATE = pd.to_datetime("2026-07-11")

# 1. Group by folder to isolate the maximum data date before/on the crash
df_outage = df_chrono.groupby("Folder")["ParsedDate"].max().reset_index()

# 2. Calculate the operational lag relative to the final outage day
df_outage["Days_Before_Crash"] = (OUTAGE_DATE - df_outage["ParsedDate"]).dt.days

# 3. Convert dates to words (e.g., July 11, 2026)
df_outage["Latest_Data_Date"] = df_outage["ParsedDate"].dt.strftime('%B %d, %Y')

# 4. Separate the folders into clear status groups
zombies = df_outage[df_outage["Days_Before_Crash"] > 2]
lagging = df_outage[(df_outage["Days_Before_Crash"] <= 2) & (df_outage["Days_Before_Crash"] > 0)]
active_at_crash = df_outage[df_outage["Days_Before_Crash"] == 0]

print("❌ CRITICAL STALE FOLDERS (Dead long before the crash):")
print(zombies[["Folder", "Latest_Data_Date", "Days_Before_Crash"]].to_string(index=False))

print("\n⚠️ LAGGING STREAMS (Stopped 1-2 days early):")
print(lagging[["Folder", "Latest_Data_Date", "Days_Before_Crash"]].to_string(index=False))

print("\n✅ ACTIVE UNTIL CRASH (Went down with the server on July 11):")
print(active_at_crash[["Folder", "Latest_Data_Date"]].to_string(index=False))


## publishing frequency

In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from collections import defaultdict

# 1. Efficient In-Memory Dictionary Storage
folder_dates = defaultdict(set)
total_folders = len(folder_profiles)

print(f"🚀 Calculating publishing frequencies across {total_folders} folders...\n")

# 2. Fast Single-Pass Network Extract
for index, f in enumerate(folder_profiles, start=1):
    folder_path = f.replace(BASE_URL, "/")
    print(f"[{index}/{total_folders}] Analyzing: {folder_path} ...", end="\r", flush=True)
    
    try:
        response_text = requests.get(f, auth=AUTH, timeout=10).text
        soup = BeautifulSoup(response_text, "html.parser")
        
        for l in soup.find_all("a"):
            filename = l.text.strip()
            date_match = re.search(r'_(\d{8})', filename)
            
            if date_match:
                # Store raw strings in a set to avoid processing duplicate dates on same day
                folder_dates[folder_path].add(date_match.group(1))
    except Exception:
        pass

print("\n\n✅ Extraction complete. Computing frequencies...")

# 3. Blazing Fast Frequency Computation Loop
report_rows = []
for folder, date_strings in sorted(folder_dates.items()):
    if len(date_strings) < 2:
        frequency = "Static / Insufficient Data"
    else:
        # Convert unique dates to sorted pandas datetime objects
        timestamps = sorted(pd.to_datetime(list(date_strings), format='%Y%m%d'))
        
        # Calculate time difference between consecutive file uploads
        intervals = [(timestamps[i] - timestamps[i-1]).days for i in range(1, len(timestamps))]
        avg_gap = sum(intervals) / len(intervals)
        
        # Mathematically match average gap to true ingestion frequencies
        if avg_gap <= 0.15: # Multiple files per day
            frequency = "Sub-Daily (Hourly updates)"
        elif 0.85 <= avg_gap <= 1.15:
            frequency = "Daily"
        elif 6.0 <= avg_gap <= 8.0:
            frequency = "Weekly"
        elif 26.0 <= avg_gap <= 32.0:
            frequency = "Monthly"
        else:
            frequency = f"Irregular (Avg gap: {avg_gap:.1f} days)"
            
    report_rows.append({
        "Folder Path": folder,
        "Unique Date Samples": len(date_strings),
        "Publishing Frequency": frequency
    })

# 4. Display the Output Presentation Table
df_freq = pd.DataFrame(report_rows)
print("\n" + "="*80)
print(" 📊 SERVER PIPELINE PUBLISHING FREQUENCIES")
print("="*80)
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(df_freq.to_string(index=False))
print("="*80 + "\n")


🚀 Calculating publishing frequencies across 19 folders...

[19/19] Analyzing: /HWRF/HWRF_MoM/ ... .....

✅ Extraction complete. Computing frequencies...

 📊 SERVER PIPELINE PUBLISHING FREQUENCIES
          Folder Path  Unique Date Samples          Publishing Frequency
        /DFO/DFO_MoM/                 1659                         Daily
      /DFO/DFO_image/                    3                         Daily
    /DFO/DFO_summary/                 1663                         Daily
        /Final_Alert/                 1653                         Daily
      /GFMS/GFMS_MoM/                 1668                         Daily
    /GFMS/GFMS_image/                    3 Irregular (Avg gap: 1.5 days)
  /GFMS/GFMS_summary/                 1670                         Daily
             /GLOFAS/                 1668                         Daily
      /HWRF/HWRF_MoM/                 1666                         Daily
    /VIIRS/VIIRS_MoM/                 1623                         Daily
/

## integrity checking

In [41]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

TARGET_MOM_FOLDER = "/GFMS/GFMS_MoM/"
target_url = f"{BASE_URL.rstrip('/')}{TARGET_MOM_FOLDER}"

try:
    response = requests.get(target_url, auth=AUTH, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    
    all_filenames = [link.text.strip() for link in soup.find_all("a") 
                     if link.text.strip() and link.text.strip().endswith(".csv") and not link.text.strip().startswith("?")]
    
    # 1. Isolate the files into clean structural sets
    pattern_clean = re.compile(r'^Attributes_Clean_(\d{8})\.csv$')
    pattern_final = re.compile(r'^Final_Attributes_(\d{8})\.csv$')
    
    clean_file_registry = {}
    final_file_registry = {}
    
    for name in all_filenames:
        match_clean = pattern_clean.match(name)
        if match_clean:
            # Map Date -> Actual Filename
            clean_file_registry[match_clean.group(1)] = name
            continue
            
        match_final = pattern_final.match(name)
        if match_final:
            # Map Date -> Actual Filename
            final_file_registry[match_final.group(1)] = name

    # 2. Build the explicit 1-to-1 Map Matrix
    paired_file_mapping = {}
    unpaired_clean_files = []
    
    # Loop over every single identified clean file registry entry
    for date_key, clean_filename in clean_file_registry.items():
        # Check if the date explicitly maps to a sister final file
        if date_key in final_file_registry:
            paired_file_mapping[date_key] = {
                "Clean_Source_File": clean_filename,
                "Final_Output_File": final_file_registry[date_key]
            }
        else:
            unpaired_clean_files.append(clean_filename)

    # 3. Check for any final files left stranded without a clean source file
    unpaired_final_files = [fname for d_key, fname in final_file_registry.items() if d_key not in clean_file_registry]

    # --- RENDER STRATIFIED STRUCTURAL MAPPING VIEW ---
    print("\n" + "="*75)
    print(" 🔗 EXPLICIT 1-TO-1 DATE MAPPING MATRIX VERIFICATION")
    print("="*75)
    print(f" Successfully Mapped File Pairs : {len(paired_file_mapping)}")
    print(f" Broken Clean Source Links     : {len(unpaired_clean_files)}")
    print(f" Broken Final Output Links     : {len(unpaired_final_files)}")
    print("-"*75)
    
    if len(unpaired_clean_files) == 0 and len(unpaired_final_files) == 0:
        print(" ✅ VERIFICATION SUCCESS: 100% of dates map to their exact sister file.")
        
        # Display a quick snapshot of the actual structural map connection in memory
        sample_dates = sorted(list(paired_file_mapping.keys()))[:2]
        print("\n 📋 Relational Mapping Dictionary Sample:")
        for s_date in sample_dates:
            print(f"   Date {s_date} ──►")
            print(f"     ├── Clean File: {paired_file_mapping[s_date]['Clean_Source_File']}")
            print(f"     └── Final File: {paired_file_mapping[s_date]['Final_Output_File']}")
    else:
        print(" ❌ MAPPING INTEGRITY CRASH: Disconnected files found.")
        if unpaired_clean_files:
            print(f"   └── Stranded Clean Files: {unpaired_clean_files[:2]}")
        if unpaired_final_files:
            print(f"   └── Stranded Final Files: {unpaired_final_files[:2]}")
    print("="*75 + "\n")

except Exception as e:
    print(f"❌ Error compiling explicit mapping dictionary: {e}")



 🔗 EXPLICIT 1-TO-1 DATE MAPPING MATRIX VERIFICATION
 Successfully Mapped File Pairs : 1668
 Broken Clean Source Links     : 0
 Broken Final Output Links     : 0
---------------------------------------------------------------------------
 ✅ VERIFICATION SUCCESS: 100% of dates map to their exact sister file.

 📋 Relational Mapping Dictionary Sample:
   Date 20211214 ──►
     ├── Clean File: Attributes_Clean_20211214.csv
     └── Final File: Final_Attributes_20211214.csv
   Date 20211216 ──►
     ├── Clean File: Attributes_Clean_20211216.csv
     └── Final File: Final_Attributes_20211216.csv

